In [ ]:
# In[1]:

import pandas as pd

# Load CSV files
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (per rule 10) and keep original Unix seconds min/max
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# --- metric.csv aggregates ---
metric_total_rows = metric_df.shape[0]
metric_min_ts = int(metric_df['timestamp'].min()) if metric_total_rows>0 else None
metric_max_ts = int(metric_df['timestamp'].max()) if metric_total_rows>0 else None
metric_unique_cmdb = pd.DataFrame({'cmdb_id': pd.Series(metric_df['cmdb_id'].unique()[:50])})
metric_kpi_counts = (
    metric_df['kpi_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index':'kpi_name', 'kpi_name':'count'})
    .head(50)
)
metric_top_pairs = (
    metric_df
    .groupby(['cmdb_id','kpi_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

metric_overview = pd.DataFrame([{
    'file':'metric.csv',
    'total_rows': metric_total_rows,
    'min_timestamp_unix': metric_min_ts,
    'max_timestamp_unix': metric_max_ts,
    'unique_cmdb_ids_reported': metric_unique_cmdb.shape[0]
}])

# --- trace.csv aggregates ---
trace_total_rows = trace_df.shape[0]
trace_min_ts = int(trace_df['timestamp'].min()) if trace_total_rows>0 else None
trace_max_ts = int(trace_df['timestamp'].max()) if trace_total_rows>0 else None
trace_unique_cmdb = pd.DataFrame({'cmdb_id': pd.Series(trace_df['cmdb_id'].unique()[:50])})
trace_name_counts = (
    trace_df['trace_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index':'trace_name', 'trace_name':'count'})
    .head(50)
)
trace_overview = pd.DataFrame([{
    'file':'trace.csv',
    'total_rows': trace_total_rows,
    'min_timestamp_unix': trace_min_ts,
    'max_timestamp_unix': trace_max_ts,
    'unique_cmdb_ids_reported': trace_unique_cmdb.shape[0]
}])

# --- log.csv aggregates ---
log_total_rows = log_df.shape[0]
log_min_ts = int(log_df['timestamp'].min()) if log_total_rows>0 else None
log_max_ts = int(log_df['timestamp'].max()) if log_total_rows>0 else None
log_unique_cmdb = pd.DataFrame({'cmdb_id': pd.Series(log_df['cmdb_id'].unique()[:50])})
log_name_counts = (
    log_df['log_name']
    .value_counts()
    .reset_index()
    .rename(columns={'index':'log_name', 'log_name':'count'})
    .head(50)
)
log_overview = pd.DataFrame([{
    'file':'log.csv',
    'total_rows': log_total_rows,
    'min_timestamp_unix': log_min_ts,
    'max_timestamp_unix': log_max_ts,
    'unique_cmdb_ids_reported': log_unique_cmdb.shape[0]
}])

# --- error_logs.csv aggregates ---
error_total_rows = error_df.shape[0]
error_min_ts = int(error_df['timestamp'].min()) if error_total_rows>0 else None
error_max_ts = int(error_df['timestamp'].max()) if error_total_rows>0 else None
error_unique_cmdb = pd.DataFrame({'cmdb_id': pd.Series(error_df['cmdb_id'].unique()[:50])})
# Per instruction: return count of distinct messages if available
distinct_error_messages = int(error_df['message'].nunique()) if 'message' in error_df.columns else None
# Also list up to first 50 distinct messages (compact)
error_messages_sample = (
    pd.DataFrame({'message': pd.Series(error_df['message'].dropna().unique()[:50])})
    if 'message' in error_df.columns else pd.DataFrame({'message': []})
)
error_overview = pd.DataFrame([{
    'file':'error_logs.csv',
    'total_rows': error_total_rows,
    'min_timestamp_unix': error_min_ts,
    'max_timestamp_unix': error_max_ts,
    'unique_cmdb_ids_reported': error_unique_cmdb.shape[0],
    'distinct_messages_count': distinct_error_messages
}])

# Display compact summaries (DataFrames only, limited rows)
metric_overview, metric_unique_cmdb, metric_kpi_counts, metric_top_pairs, \
trace_overview, trace_unique_cmdb, trace_name_counts, \
log_overview, log_unique_cmdb, log_name_counts, \
error_overview, error_unique_cmdb, error_messages_sample

```
Out[1]:
```
```python
import pandas as pd

# Build a concise plain-English summary from existing variables
m_over = metric_overview.iloc[0]
m_total = int(m_over['total_rows'])
m_min = m_over['min_timestamp_unix']
m_max = m_over['max_timestamp_unix']
m_min_iso = pd.to_datetime(int(m_min), unit='s', utc=True).isoformat() if pd.notna(m_min) else "N/A"
m_max_iso = pd.to_datetime(int(m_max), unit='s', utc=True).isoformat() if pd.notna(m_max) else "N/A"
metric_cmdb_list = metric_unique_cmdb['cmdb_id'].astype(str).tolist()[:10]
metric_kpi_top = metric_df['kpi_name'].value_counts().head(6)

top_pairs = metric_top_pairs.head(10)
top_pairs_str = "; ".join([f"{r['cmdb_id']}/{r['kpi_name']}={int(r['count'])}" for _, r in top_pairs.iterrows()])

trace_total = int(trace_overview.iloc[0]['total_rows'])

log_over = log_overview.iloc[0]
log_total = int(log_over['total_rows'])
log_cmdb_list = log_unique_cmdb['cmdb_id'].astype(str).tolist()[:10]
log_kinds = log_df['log_name'].value_counts().head(5)

error_total = int(error_overview.iloc[0]['total_rows'])
error_distinct_msgs = int(error_overview.iloc[0]['distinct_messages_count']) if 'distinct_messages_count' in error_overview.columns else 0

summary = (
    f"Telemetry summary (plain English):\n"
    f"- metric.csv: {m_total} rows across {int(m_over['unique_cmdb_ids_reported'])} cmdb_ids. "
    f"Timestamps range from {m_min} (UTC {m_min_iso}) to {m_max} (UTC {m_max_iso}).\n"
    f"  Sample cmdb_ids (up to 10): {', '.join(metric_cmdb_list)}.\n"
    f"  Top KPI counts: " + "; ".join([f"{k}={v}" for k,v in metric_kpi_top.items()]) + ".\n"
    f"  Top (cmdb_id, kpi_name) pairs (top 10 by rows): {top_pairs_str}.\n\n"
    f"- trace.csv: {trace_total} rows (no trace data available).\n\n"
    f"- log.csv: {log_total} rows across {int(log_over['unique_cmdb_ids_reported'])} cmdb_ids. "
    f"Sample cmdb_ids (up to 10): {', '.join(log_cmdb_list)}.\n"
    f"  Log types/counts (top): " + "; ".join([f\"{k}={v}\" for k,v in log_kinds.items()]) + ".\n\n"
    f"- error_logs.csv: {error_total} rows, distinct messages: {error_distinct_msgs}.\n\n"
    f"Overall: metrics are present for many services (16 cmdb_ids) with CPU, memory, and socket metrics most frequent; "
    f"logs exist for several services but there are no error_log entries and no trace telemetry in this dataset."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(         file  total_rows  min_timestamp_unix  max_timestamp_unix  unique_cmdb_ids_reported
0  metric.csv        1853          1705727940          1705729380                        16,               cmdb_id
0               carts
1            carts-db
2           catalogue
3        catalogue-db
4           front-end
5          istio-init
6              orders
7           orders-db
8             payment
9        queue-master
10           rabbitmq
11  rabbitmq-exporter
12         session-db
13           shipping
14               user
15            user-db,         count  count
0         mem    377
1      socket    377
2         cpu    375
3      diskio    199
4  latency-50    175
5  latency-90    175
6    workload    175,          cmdb_id    kpi_name  count
0          carts         cpu     25
1          carts      diskio     25
2          carts  latency-50     25
3          carts  latency-90     25
4          carts         mem     25
5          carts      socket     25
6          carts    workload     25
7       carts-db         cpu     25
8       carts-db      diskio     25
9       carts-db         mem     25
10      carts-db      socket     25
11     catalogue         cpu     25
12     catalogue  latency-50     25
13     catalogue  latency-90     25
14     catalogue         mem     25
15     catalogue      socket     25
16     catalogue    workload     25
17  catalogue-db         cpu     25
18  catalogue-db      diskio     25
19  catalogue-db         mem     25,         file  total_rows min_timestamp_unix max_timestamp_unix  unique_cmdb_ids_reported
0  trace.csv           0               None               None                         0, Empty DataFrame
Columns: [cmdb_id]
Index: [], Empty DataFrame
Columns: [count, count]
Index: [],       file  total_rows  min_timestamp_unix  max_timestamp_unix  unique_cmdb_ids_reported
0  log.csv         454          1705727940          1705729380                        11,          cmdb_id
0          carts
1      catalogue
2      front-end
3         orders
4        payment
5   queue-master
6       shipping
7           user
8       carts-db
9     session-db
10       user-db,              count  count
0  log.error_count    227
1    log.row_count    227,              file  total_rows min_timestamp_unix max_timestamp_unix  unique_cmdb_ids_reported  distinct_messages_count
0  error_logs.csv           0               None               None                         0                        0, Empty DataFrame
Columns: [cmdb_id]
Index: [], Empty DataFrame
Columns: [message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Ensure metric_df is loaded; if not, load it
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure timestamp_dt exists and timestamp is numeric Unix seconds
if 'timestamp_dt' not in metric_df.columns:
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# 1) Compute global P95 threshold per (cmdb_id, kpi_name) using all available rows
thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(lambda s: float(s.quantile(0.95)))
    .reset_index()
    .rename(columns={'value': 'global_p95_threshold'})
)

# 2) Filter metric rows to the incident window (inclusive)
start_ts = pd.to_datetime('2024-01-20 05:16:13', utc=True)
end_ts = pd.to_datetime('2024-01-20 05:46:13', utc=True)
start_unix = int(start_ts.timestamp())
end_unix = int(end_ts.timestamp())

metric_window = metric_df[(metric_df['timestamp'] >= start_unix) & (metric_df['timestamp'] <= end_unix)].copy()

# 3) Merge thresholds into windowed rows to evaluate anomalies (value >= global_p95_threshold)
window_merged = metric_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')
window_merged['is_anomaly'] = window_merged['value'] >= window_merged['global_p95_threshold']

# Aggregations for pairs that have rows in the window
total_points_series = window_merged.groupby(['cmdb_id','kpi_name'])['value'].size().rename('total_points_in_window')
anomaly_counts_series = window_merged.groupby(['cmdb_id','kpi_name'])['is_anomaly'].sum().rename('anomaly_count_in_window')
earliest_anomaly_series = (
    window_merged[window_merged['is_anomaly']]
    .groupby(['cmdb_id','kpi_name'])['timestamp']
    .min()
    .rename('earliest_anomaly_timestamp_unix')
)
max_value_series = window_merged.groupby(['cmdb_id','kpi_name'])['value'].max().rename('max_value_in_window')

# Start from the full set of distinct pairs (thresholds) to include pairs with zero points in window
result_df = thresholds.merge(total_points_series.reset_index(), on=['cmdb_id','kpi_name'], how='left')
result_df = result_df.merge(anomaly_counts_series.reset_index(), on=['cmdb_id','kpi_name'], how='left')
result_df = result_df.merge(earliest_anomaly_series.reset_index(), on=['cmdb_id','kpi_name'], how='left')
result_df = result_df.merge(max_value_series.reset_index(), on=['cmdb_id','kpi_name'], how='left')

# Fill NaNs for counts with 0, keep earliest_anomaly_timestamp_unix as NA if none, keep max_value as NA if none
result_df['total_points_in_window'] = result_df['total_points_in_window'].fillna(0).astype(int)
result_df['anomaly_count_in_window'] = result_df['anomaly_count_in_window'].fillna(0).astype(int)
# Convert earliest anomaly timestamps to nullable integer type for compact display (or keep NaN)
result_df['earliest_anomaly_timestamp_unix'] = result_df['earliest_anomaly_timestamp_unix'].astype('Int64')
# max_value_in_window keep as float with NaN for no-data pairs

# 4) Return top 20 rows sorted by anomaly_count_in_window desc, then total_points_in_window desc
result_df_sorted = result_df.sort_values(
    by=['anomaly_count_in_window', 'total_points_in_window'],
    ascending=[False, False]
).reset_index(drop=True)

top20_result = result_df_sorted.head(20)[[
    'cmdb_id',
    'kpi_name',
    'global_p95_threshold',
    'total_points_in_window',
    'anomaly_count_in_window',
    'earliest_anomaly_timestamp_unix',
    'max_value_in_window'
]]

# Also include total number of distinct (cmdb_id, kpi_name) pairs processed
total_pairs = int(thresholds.shape[0])

# Display the compact outputs (variables only)
top20_result, total_pairs

```
Out[2]:
```
```python
# Build a concise plain-English summary from the previously computed results
try:
    top20 = top20_result
    total = total_pairs
except NameError:
    raise RuntimeError("Expected variables top20_result and total_pairs in kernel state.")

lines = []
lines.append(f"Processed {total} distinct (cmdb_id, kpi_name) pairs.")
lines.append("Top 20 pairs sorted by anomaly_count_in_window (desc), then total_points_in_window:")
for i, r in top20.reset_index(drop=True).iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    p95 = r['global_p95_threshold']
    total_pts = int(r['total_points_in_window'])
    anomalies = int(r['anomaly_count_in_window'])
    earliest_ts = r['earliest_anomaly_timestamp_unix']
    if pd.isna(earliest_ts):
        earliest_str = "null"
    else:
        # earliest_ts may be Int64 or int
        earliest_str = pd.to_datetime(int(earliest_ts), unit='s', utc=True).isoformat()
    max_val = r['max_value_in_window']
    # concise formatting for numeric values
    p95_s = f"{p95:.6g}" if pd.notna(p95) else "null"
    max_s = f"{max_val:.6g}" if pd.notna(max_val) else "null"
    lines.append(
        f"{i+1}. {cmdb} / {kpi} | anomalies={anomalies} | points={total_pts} | "
        f"earliest_anomaly={earliest_str} | global_p95={p95_s} | max_in_window={max_s}"
    )

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  global_p95_threshold  total_points_in_window  anomaly_count_in_window  earliest_anomaly_timestamp_unix  max_value_in_window
0               carts      diskio          0.000000e+00                      25                       25                       1705727940         0.000000e+00
1        catalogue-db      diskio          0.000000e+00                      25                       25                       1705727940         0.000000e+00
2        catalogue-db      socket          4.000000e+00                      25                       25                       1705727940         4.000000e+00
3           orders-db      socket          7.000000e+00                      25                       25                       1705727940         7.000000e+00
4        queue-master      socket          3.000000e+00                      25                       25                       1705727940         3.000000e+00
5            rabbitmq      diskio          0.000000e+00                      25                       25                       1705727940         3.940510e+03
6            rabbitmq      socket          1.100000e+01                      25                       25                       1705727940         1.100000e+01
7   rabbitmq-exporter         mem          4.644864e+06                      25                       25                       1705727940         4.644864e+06
8   rabbitmq-exporter      socket          1.000000e+00                      25                       25                       1705727940         1.000000e+00
9          session-db      socket          3.000000e+00                      25                       25                       1705727940         3.000000e+00
10          catalogue      socket          6.000000e+00                      25                       18                       1705728360         6.000000e+00
11            payment      socket          5.000000e+00                      25                       18                       1705728360         5.000000e+00
12               user      socket          1.600000e+01                      25                       17                       1705728420         1.600000e+01
13            user-db      socket          8.000000e+00                      25                       17                       1705728420         8.000000e+00
14           carts-db      socket          1.300000e+01                      25                        6                       1705729080         1.300000e+01
15              carts         cpu          5.880919e+00                      25                        2                       1705727940         1.019964e+01
16              carts  latency-50          3.804054e-01                      25                        2                       1705728780         3.958072e-01
17              carts  latency-90          1.978317e+00                      25                        2                       1705728900         1.992000e+00
18              carts         mem          2.164932e+08                      25                        2                       1705729320         2.166743e+08
19              carts      socket          2.386000e+01                      25                        2                       1705729080         2.410000e+01, 76)```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing loaded dataframes if available, else load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

# Ensure timestamp columns and datetime parsed per rule 10 (UTC)
if 'timestamp_dt' not in metric_df.columns:
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
if 'timestamp_dt' not in log_df.columns:
    log_df['timestamp_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Target components
targets = ['catalogue','payment','user','carts','orders']

# 1) Compute global P95 for each (cmdb_id, kpi_name) pair for target components using full series
metric_targets = metric_df[metric_df['cmdb_id'].isin(targets)].copy()
thresholds = (
    metric_targets
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value':'global_p95_threshold'})
)

# Incident window (inclusive)
start_ts = pd.to_datetime('2024-01-20 05:16:13', utc=True)
end_ts = pd.to_datetime('2024-01-20 05:46:13', utc=True)
start_unix = int(start_ts.timestamp())
end_unix = int(end_ts.timestamp())

# 2) Filter metric rows to the incident window for target components
metric_window = metric_df[
    (metric_df['timestamp'] >= start_unix) &
    (metric_df['timestamp'] <= end_unix) &
    (metric_df['cmdb_id'].isin(targets))
].copy()

# Merge thresholds into windowed rows to evaluate anomalies
mw = metric_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
# If a pair exists in window but somehow not in thresholds (unlikely), compute its threshold from full data fallback
# (But per rule 9 thresholds must be computed from full series first - so thresholds already contains pairs from full series)

mw['is_anomaly'] = mw['value'] >= mw['global_p95_threshold']

# Aggregations per (cmdb_id, kpi_name) for windowed data
agg_total = mw.groupby(['cmdb_id','kpi_name'])['value'].size().rename('total_points_in_window').reset_index()
agg_anom = mw.groupby(['cmdb_id','kpi_name'])['is_anomaly'].sum().rename('anomaly_count_in_window').reset_index()
agg_earliest = (
    mw[mw['is_anomaly']]
    .groupby(['cmdb_id','kpi_name'])['timestamp']
    .min()
    .rename('earliest_anomaly_timestamp_unix')
    .reset_index()
)
agg_max = mw.groupby(['cmdb_id','kpi_name'])['value'].max().rename('max_value_in_window').reset_index()

# Start from thresholds to include pairs that may have zero points in the window
per_pair = thresholds.merge(agg_total, on=['cmdb_id','kpi_name'], how='left')
per_pair = per_pair.merge(agg_anom, on=['cmdb_id','kpi_name'], how='left')
per_pair = per_pair.merge(agg_earliest, on=['cmdb_id','kpi_name'], how='left')
per_pair = per_pair.merge(agg_max, on=['cmdb_id','kpi_name'], how='left')

# Fill missing counts with zeros; keep earliest as nullable Int64; max as float NaN if no points
per_pair['total_points_in_window'] = per_pair['total_points_in_window'].fillna(0).astype(int)
per_pair['anomaly_count_in_window'] = per_pair['anomaly_count_in_window'].fillna(0).astype(int)
per_pair['earliest_anomaly_timestamp_unix'] = per_pair['earliest_anomaly_timestamp_unix'].astype('Int64')
# Reorder columns as requested
per_pair = per_pair[[
    'cmdb_id',
    'kpi_name',
    'global_p95_threshold',
    'total_points_in_window',
    'anomaly_count_in_window',
    'earliest_anomaly_timestamp_unix',
    'max_value_in_window'
]]

# Keep full intermediate table in variable per_pair_full (per rules)
per_pair_full = per_pair.copy()

# Limit per-pair output to no more than 50 rows for display
per_pair_df = per_pair_full.head(50)

# 3) Aggregate per component
def compact_kpi_list(df_comp):
    # df_comp: rows for single component with anomaly_count_in_window>0
    dfc = df_comp[df_comp['anomaly_count_in_window']>0].copy()
    if dfc.shape[0]==0:
        return ""
    dfc = dfc.sort_values('anomaly_count_in_window', ascending=False).head(10)
    pairs = dfc.apply(lambda r: f"{r['kpi_name']}:{int(r['anomaly_count_in_window'])}", axis=1)
    return ", ".join(pairs.tolist())

comp_agg = []
for comp in targets:
    comp_rows = per_pair_full[per_pair_full['cmdb_id']==comp]
    component_total_anomalies = int(comp_rows['anomaly_count_in_window'].sum())
    num_kpis_with_anomalies = int((comp_rows['anomaly_count_in_window']>0).sum())
    # earliest overall anomaly (min across KPIs), keep as Int64 nullable
    if comp_rows['earliest_anomaly_timestamp_unix'].dropna().shape[0] > 0:
        earliest_overall = int(comp_rows['earliest_anomaly_timestamp_unix'].dropna().min())
    else:
        earliest_overall = pd.NA
    kpi_list = compact_kpi_list(comp_rows)
    comp_agg.append({
        'cmdb_id': comp,
        'component_total_anomalies': component_total_anomalies,
        'num_kpis_with_anomalies': num_kpis_with_anomalies,
        'earliest_anomaly_overall_unix': pd.Int64Dtype().type(earliest_overall) if earliest_overall is not pd.NA else pd.NA,
        'kpi_anomaly_list_top10': kpi_list
    })
per_component_df = pd.DataFrame(comp_agg)
# Ensure earliest_anomaly_overall_unix is Int64 nullable
per_component_df['earliest_anomaly_overall_unix'] = per_component_df['earliest_anomaly_overall_unix'].astype('Int64')

# 4) Log.csv processing for same incident window per component
log_window = log_df[
    (log_df['timestamp'] >= start_unix) &
    (log_df['timestamp'] <= end_unix) &
    (log_df['cmdb_id'].isin(targets))
].copy()

# total_log_rows_in_window per component
log_total = log_window.groupby('cmdb_id').size().rename('total_log_rows_in_window').reset_index()

# total_log_error_count_rows_in_window where log_name == 'log.error_count'
log_error_rows = log_window[log_window['log_name']=='log.error_count'].copy()
log_error_total = log_error_rows.groupby('cmdb_id').size().rename('total_log_error_count_rows_in_window').reset_index()

# count_log_error_count_positive: value>0
log_error_positive = log_error_rows[log_error_rows['value']>0].copy()
log_error_positive_count = log_error_positive.groupby('cmdb_id').size().rename('count_log_error_count_positive').reset_index()

# earliest_log_error_count_positive_timestamp_unix
if not log_error_positive.empty:
    log_error_earliest = log_error_positive.groupby('cmdb_id')['timestamp'].min().rename('earliest_log_error_count_positive_timestamp_unix').reset_index()
else:
    log_error_earliest = pd.DataFrame(columns=['cmdb_id','earliest_log_error_count_positive_timestamp_unix'])

# Start from targets to ensure components with zero logs are included
log_comp_df = pd.DataFrame({'cmdb_id': targets})
log_comp_df = log_comp_df.merge(log_total, on='cmdb_id', how='left')
log_comp_df = log_comp_df.merge(log_error_total, on='cmdb_id', how='left')
log_comp_df = log_comp_df.merge(log_error_positive_count, on='cmdb_id', how='left')
log_comp_df = log_comp_df.merge(log_error_earliest, on='cmdb_id', how='left')

# Fill NaNs with zeros for counts and keep earliest as Int64 nullable
log_comp_df['total_log_rows_in_window'] = log_comp_df['total_log_rows_in_window'].fillna(0).astype(int)
log_comp_df['total_log_error_count_rows_in_window'] = log_comp_df['total_log_error_count_rows_in_window'].fillna(0).astype(int)
log_comp_df['count_log_error_count_positive'] = log_comp_df['count_log_error_count_positive'].fillna(0).astype(int)
log_comp_df['earliest_log_error_count_positive_timestamp_unix'] = log_comp_df['earliest_log_error_count_positive_timestamp_unix'].astype('Int64')

# Merge log summary into per_component_df
per_component_df = per_component_df.merge(log_comp_df, on='cmdb_id', how='left')

# Final outputs: (A) per-(cmdb_id,kpi_name) rows (limit to <=50 rows) (B) per-component aggregates (one row per component)
# Store full intermediate tables in kernel variables: per_pair_full, per_component_df, log_window, metric_window
per_pair_df, per_component_df

```
Out[3]:
```
import pandas as pd

# Build a plain-English summary string using the existing kernel variables
try:
    per_pair = per_pair_df      # per-(cmdb_id,kpi_name) table (limited view)
    per_comp = per_component_df # per-component aggregates
except NameError:
    raise RuntimeError("Expected per_pair_df and per_component_df to be present in kernel state.")

lines = []
lines.append(f"Summary for target components ['catalogue','payment','user','carts','orders']:")
lines.append(f"- Number of (cmdb_id, kpi_name) pairs reported (displayed): {per_pair.shape[0]}")
lines.append("")

lines.append("Per-component aggregates:")
for _, r in per_comp.iterrows():
    comp = r['cmdb_id']
    total_anom = int(r['component_total_anomalies'])
    num_kpis = int(r['num_kpis_with_anomalies'])
    earliest_unix = r['earliest_anomaly_overall_unix']
    if pd.isna(earliest_unix):
        earliest_str = "null"
    else:
        earliest_str = pd.to_datetime(int(earliest_unix), unit='s', utc=True).isoformat()
    kpi_list = r['kpi_anomaly_list_top10'] if r['kpi_anomaly_list_top10'] else "(none)"
    # log fields
    total_log_rows = int(r.get('total_log_rows_in_window', 0))
    total_log_error_rows = int(r.get('total_log_error_count_rows_in_window', 0))
    count_log_error_positive = int(r.get('count_log_error_count_positive', 0))
    earliest_log_err_pos = r.get('earliest_log_error_count_positive_timestamp_unix', pd.NA)
    if pd.isna(earliest_log_err_pos):
        earliest_log_err_pos_str = "null"
    else:
        earliest_log_err_pos_str = pd.to_datetime(int(earliest_log_err_pos), unit='s', utc=True).isoformat()

    lines.append(
        f"- {comp}: total_anomalies={total_anom}, kpis_with_anomalies={num_kpis}, "
        f"earliest_anomaly={earliest_str}; top KPI:counts [{kpi_list}]."
    )
    lines.append(
        f"  Logs in window: total_rows={total_log_rows}, log.error_count_rows={total_log_error_rows}, "
        f"log.error_count>0 count={count_log_error_positive}, earliest_positive_log_error={earliest_log_err_pos_str}."
    )

lines.append("")
lines.append("Notes:")
lines.append("- Thresholds (global P95) were computed from the full series for each (cmdb_id,kpi_name).")
lines.append("- Metric anomalies are points within 2024-01-20 05:16:13 to 2024-01-20 05:46:13 UTC where value >= global P95.")
lines.append("- No log.error_count>0 occurrences were found in the window for these components (count_log_error_count_positive = 0).")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id    kpi_name  global_p95_threshold  total_points_in_window  anomaly_count_in_window  earliest_anomaly_timestamp_unix  max_value_in_window
0    carts         cpu          5.880919e+00                      25                        2                       1705727940         1.019964e+01
1    carts      diskio          0.000000e+00                      25                       25                       1705727940         0.000000e+00
2    carts  latency-50          3.804054e-01                      25                        2                       1705728780         3.958072e-01
3    carts  latency-90          1.978317e+00                      25                        2                       1705728900         1.992000e+00
4    carts         mem          2.164932e+08                      25                        2                       1705729320         2.166743e+08
..     ...         ...                   ...                     ...                      ...                              ...                  ...
26    user  latency-50          3.342191e-03                      25                        2                       1705728060         3.349130e-03
27    user  latency-90          6.565824e-03                      25                        2                       1705728060         6.632846e-03
28    user         mem          7.421215e+06                      25                        2                       1705729320         7.435410e+06
29    user      socket          1.600000e+01                      25                       17                       1705728420         1.600000e+01
30    user    workload          1.921468e+01                      25                        2                       1705728060         1.957685e+01

[31 rows x 7 columns],      cmdb_id  component_total_anomalies  num_kpis_with_anomalies  earliest_anomaly_overall_unix                             kpi_anomaly_list_top10  total_log_rows_in_window  total_log_error_count_rows_in_window  count_log_error_count_positive  earliest_log_error_count_positive_timestamp_unix
0  catalogue                         28                        6                     1705728060  socket:18, cpu:2, latency-50:2, latency-90:2, ...                        50                                    25                               0                                              <NA>
1    payment                         28                        6                     1705727940  socket:18, cpu:2, latency-50:2, latency-90:2, ...                        50                                    25                               0                                              <NA>
2       user                         27                        6                     1705728060  socket:17, cpu:2, latency-50:2, latency-90:2, ...                        50                                    25                               0                                              <NA>
3      carts                         37                        7                     1705727940  diskio:25, cpu:2, latency-50:2, latency-90:2, ...                        40                                    20                               0                                              <NA>
4     orders                         12                        6                     1705727940  cpu:2, latency-50:2, latency-90:2, mem:2, sock...                        50                                    25                               0                                              <NA>)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing variables when available
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

targets = ['catalogue','payment','user','carts','orders']

# Ensure timestamps parsed and available
if 'timestamp_dt' not in metric_df.columns:
    metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure global thresholds for target components exist (thresholds variable)
try:
    thresholds
except NameError:
    metric_targets = metric_df[metric_df['cmdb_id'].isin(targets)].copy()
    thresholds = (
        metric_targets
        .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
        .quantile(0.95)
        .reset_index()
        .rename(columns={'value':'global_p95_threshold'})
    )

# Ensure metric_window (filtered to incident window and targets) exists
start_ts = pd.to_datetime('2024-01-20 05:16:13', utc=True)
end_ts = pd.to_datetime('2024-01-20 05:46:13', utc=True)
start_unix = int(start_ts.timestamp())
end_unix = int(end_ts.timestamp())

try:
    metric_window
except NameError:
    metric_window = metric_df[
        (metric_df['timestamp'] >= start_unix) &
        (metric_df['timestamp'] <= end_unix) &
        (metric_df['cmdb_id'].isin(targets))
    ].copy()

# Merge thresholds into the windowed metrics
mw = metric_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# Mark anomalies: value >= global_p95_threshold
mw['is_anomaly'] = mw['value'] >= mw['global_p95_threshold']

# Keep anomalies dataframe (intermediate)
anomalies_df = mw[mw['is_anomaly']].copy().sort_values(['cmdb_id','kpi_name','timestamp'])

# Detect consecutive runs (consecutive minutes: diff == 60 seconds)
if not anomalies_df.empty:
    anomalies_df['ts_diff'] = anomalies_df.groupby(['cmdb_id','kpi_name'])['timestamp'].diff().fillna(9999)
    anomalies_df['new_run'] = anomalies_df['ts_diff'] != 60
    anomalies_df['run_id'] = anomalies_df.groupby(['cmdb_id','kpi_name'])['new_run'].cumsum()
    # Aggregate runs
    run_agg = (
        anomalies_df
        .groupby(['cmdb_id','kpi_name','run_id'], dropna=False)
        .agg(run_length=('value','size'),
             run_start=('timestamp','min'),
             run_max=('value','max'))
        .reset_index()
    )
else:
    # empty run_agg
    run_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','run_id','run_length','run_start','run_max'])

# Per-pair run summaries
runs_per_pair = (
    run_agg
    .groupby(['cmdb_id','kpi_name'], dropna=False)
    .agg(
        num_runs=('run_id','nunique'),
        longest_run_length=('run_length','max'),
        earliest_run_start_unix=('run_start','min')
    )
    .reset_index()
)

# Ensure pairs with zero anomalies are present: start from thresholds
pairs = thresholds[['cmdb_id','kpi_name','global_p95_threshold']].copy()

# Max value in window per pair
max_in_window = (
    metric_window
    .groupby(['cmdb_id','kpi_name'], dropna=False)['value']
    .max()
    .reset_index()
    .rename(columns={'value':'max_value_in_window'})
)

# Merge everything
summary = pairs.merge(runs_per_pair, on=['cmdb_id','kpi_name'], how='left')
summary = summary.merge(max_in_window, on=['cmdb_id','kpi_name'], how='left')

# Fill defaults for pairs without runs
summary['num_runs'] = summary['num_runs'].fillna(0).astype(int)
# longest_run_length NaN -> 0
summary['longest_run_length'] = summary['longest_run_length'].fillna(0).astype(int)
# earliest_run_start_unix keep as Int64 nullable
summary['earliest_run_start_unix'] = summary['earliest_run_start_unix'].astype('Int64')
# max_value_in_window may be NaN for no-window points

# Compute breach_ratio per rules
def compute_breach(row):
    thr = row['global_p95_threshold']
    mx = row['max_value_in_window']
    if pd.isna(mx):
        return pd.NA
    if pd.isna(thr):
        return pd.NA
    try:
        if thr > 0:
            return (mx - thr) / thr
        else:
            return mx
    except Exception:
        return pd.NA

summary['breach_ratio'] = summary.apply(compute_breach, axis=1)

# Rename 'longest_run_length' column consistent with requested output name
# Ensure column order: cmdb_id, kpi_name, num_runs, longest_run_length, earliest_run_start_unix, global_p95_threshold, max_value_in_window, breach_ratio
result_df = summary[[
    'cmdb_id',
    'kpi_name',
    'num_runs',
    'longest_run_length',
    'earliest_run_start_unix',
    'global_p95_threshold',
    'max_value_in_window',
    'breach_ratio'
]].copy()

# Sort by breach_ratio desc (NaNs at end), then longest_run_length desc
result_sorted = result_df.sort_values(
    by=['breach_ratio','longest_run_length'],
    ascending=[False, False],
    na_position='last'
).reset_index(drop=True)

# Top 20 rows
top20 = result_sorted.head(20).copy()

# Make earliest_run_start_unix Int64 for compact display
top20['earliest_run_start_unix'] = top20['earliest_run_start_unix'].astype('Int64')

# Totals: number of pairs processed and total anomalous runs across all pairs
total_pairs = int(pairs.shape[0])
total_anomalous_runs = int(run_agg.shape[0])

# Store intermediates in kernel variables (already set): anomalies_df, run_agg, runs_per_pair, summary
anomalies_df, run_agg, runs_per_pair, summary, top20, total_pairs, total_anomalous_runs

```
Out[4]:
```
```python
# Build a concise plain-English summary from previously computed results
import pandas as pd

try:
    top = top20.copy()
    total_pairs = total_pairs
    total_runs = total_anomalous_runs
except NameError:
    raise RuntimeError("Expected variables top20, total_pairs, total_anomalous_runs in kernel state.")

lines = []
lines.append(f"Summary of anomaly-run analysis for components ['catalogue','payment','user','carts','orders']:")
lines.append(f"- Processed {int(total_pairs)} distinct (cmdb_id, kpi_name) pairs.")
lines.append(f"- Detected a total of {int(total_runs)} anomalous runs across all pairs in the incident window.")
lines.append("")

# Top offenders by breach_ratio (show top 6 for a compact view)
lines.append("Top offending (cmdb_id / kpi_name) by breach ratio:")
show_n = min(6, top.shape[0])
for i, r in top.head(show_n).iterrows():
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    br = r['breach_ratio']
    br_s = f"{br:.3f}" if pd.notna(br) else "null"
    thr = r['global_p95_threshold']
    thr_s = f"{thr:.6g}" if pd.notna(thr) else "null"
    mx = r['max_value_in_window']
    mx_s = f"{mx:.6g}" if pd.notna(mx) else "null"
    lr = int(r['longest_run_length'])
    nr = int(r['num_runs'])
    ers = r['earliest_run_start_unix']
    if pd.isna(ers):
        ers_s = "null"
    else:
        ers_s = pd.to_datetime(int(ers), unit='s', utc=True).isoformat()
    lines.append(f"{i+1}. {cmdb}/{kpi} | breach_ratio={br_s} | runs={nr} (longest={lr}) | earliest_run_start={ers_s} | p95={thr_s} | max_in_window={mx_s}")

lines.append("")
lines.append("Notable observations:")
lines.append("- orders/cpu shows the largest breach_ratio (~0.958), with max ~13.37 vs global P95 ~6.83 (two short runs).")
lines.append("- carts/cpu is the second largest breach (~0.734).")
lines.append("- Some series have long consecutive runs but low breach_ratio: e.g., carts/diskio had a 25-point consecutive run (values and P95 are zero, breach_ratio=0).")
lines.append("- user/socket had a long run (17 minutes) but breach_ratio is 0 (max equals P95).")
lines.append("")
lines.append("Notes:")
lines.append("- breach_ratio = (max_value_in_window - global_p95)/global_p95 when P95>0; if P95==0, breach_ratio = max_value_in_window (handled accordingly).")
lines.append("- Timestamps shown are UTC and refer to run start times within 2024-01-20 05:16:13 to 05:46:13 UTC.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      timestamp cmdb_id  kpi_name      value              timestamp_dt  global_p95_threshold  is_anomaly  ts_diff  new_run  run_id
0    1705727940   carts       cpu  10.199635 2024-01-20 05:19:00+00:00              5.880919        True   9999.0     True       1
62   1705728060   carts       cpu   6.011248 2024-01-20 05:21:00+00:00              5.880919        True    120.0     True       2
1    1705727940   carts    diskio   0.000000 2024-01-20 05:19:00+00:00              0.000000        True   9999.0     True       1
32   1705728000   carts    diskio   0.000000 2024-01-20 05:20:00+00:00              0.000000        True     60.0    False       1
63   1705728060   carts    diskio   0.000000 2024-01-20 05:21:00+00:00              0.000000        True     60.0    False       1
..          ...     ...       ...        ...                       ...                   ...         ...      ...      ...     ...
711  1705729260    user    socket  16.000000 2024-01-20 05:41:00+00:00             16.000000        True     60.0    False       1
742  1705729320    user    socket  16.000000 2024-01-20 05:42:00+00:00             16.000000        True     60.0    False       1
773  1705729380    user    socket  16.000000 2024-01-20 05:43:00+00:00             16.000000        True     60.0    False       1
92   1705728060    user  workload  19.236750 2024-01-20 05:21:00+00:00             19.214677        True   9999.0     True       1
216  1705728300    user  workload  19.576850 2024-01-20 05:25:00+00:00             19.214677        True    240.0     True       2

[132 rows x 10 columns],    cmdb_id    kpi_name  run_id  run_length   run_start       run_max
0    carts         cpu       1           1  1705727940  1.019964e+01
1    carts         cpu       2           1  1705728060  6.011248e+00
2    carts      diskio       1          25  1705727940  0.000000e+00
3    carts  latency-50       1           1  1705728780  3.818388e-01
4    carts  latency-50       2           1  1705729020  3.958072e-01
..     ...         ...     ...         ...         ...           ...
45    user  latency-90       2           1  1705728660  6.632846e-03
46    user         mem       1           2  1705729320  7.435410e+06
47    user      socket       1          17  1705728420  1.600000e+01
48    user    workload       1           1  1705728060  1.923675e+01
49    user    workload       2           1  1705728300  1.957685e+01

[50 rows x 6 columns],    cmdb_id    kpi_name  num_runs  longest_run_length  earliest_run_start_unix
0    carts         cpu         2                   1               1705727940
1    carts      diskio         1                  25               1705727940
2    carts  latency-50         2                   1               1705728780
3    carts  latency-90         1                   2               1705728900
4    carts         mem         1                   2               1705729320
..     ...         ...       ...                 ...                      ...
26    user  latency-50         2                   1               1705728060
27    user  latency-90         2                   1               1705728060
28    user         mem         1                   2               1705729320
29    user      socket         1                  17               1705728420
30    user    workload         2                   1               1705728060

[31 rows x 5 columns],    cmdb_id    kpi_name  global_p95_threshold  num_runs  longest_run_length  earliest_run_start_unix  max_value_in_window  breach_ratio
0    carts         cpu          5.880919e+00         2                   1               1705727940         1.019964e+01      0.734361
1    carts      diskio          0.000000e+00         1                  25               1705727940         0.000000e+00      0.000000
2    carts  latency-50          3.804054e-01         2                   1               1705728780         3.958072e-01      0.040488
3    carts  latency-90          1.978317e+00         1                   2               1705728900         1.992000e+00      0.006917
4    carts         mem          2.164932e+08         1                   2               1705729320         2.166743e+08      0.000836
..     ...         ...                   ...       ...                 ...                      ...                  ...           ...
26    user  latency-50          3.342191e-03         2                   1               1705728060         3.349130e-03      0.002076
27    user  latency-90          6.565824e-03         2                   1               1705728060         6.632846e-03      0.010208
28    user         mem          7.421215e+06         1                   2               1705729320         7.435410e+06      0.001913
29    user      socket          1.600000e+01         1                  17               1705728420         1.600000e+01      0.000000
30    user    workload          1.921468e+01         2                   1               1705728060         1.957685e+01      0.018849

[31 rows x 8 columns],       cmdb_id    kpi_name  num_runs  longest_run_length  earliest_run_start_unix  global_p95_threshold  max_value_in_window  breach_ratio
0      orders         cpu         2                   1               1705727940          6.830278e+00         1.337247e+01      0.957821
1       carts         cpu         2                   1               1705727940          5.880919e+00         1.019964e+01      0.734361
2      orders  latency-50         1                   2               1705728780          2.054920e-01         2.471989e-01      0.202961
3      orders  latency-90         2                   1               1705728720          1.388143e+00         1.513526e+00      0.090324
4   catalogue         cpu         1                   2               1705728600          1.739893e-01         1.871299e-01      0.075525
5      orders    workload         2                   1               1705728300          2.116733e+00         2.216583e+00      0.047172
6       carts  latency-50         2                   1               1705728780          3.804054e-01         3.958072e-01      0.040488
7     payment    workload         2                   1               1705728300          2.111713e+00         2.196583e+00      0.040190
8     payment  latency-50         2                   1               1705727940          2.675245e-03         2.767874e-03      0.034624
9     payment  latency-90         2                   1               1705727940          4.589036e-03         4.684169e-03      0.020730
10       user    workload         2                   1               1705728060          1.921468e+01         1.957685e+01      0.018849
11    payment         mem         1                   2               1705728300          5.144890e+06         5.232503e+06      0.017029
12      carts    workload         2                   1               1705728060          8.558223e+00         8.696733e+00      0.016184
13    payment         cpu         1                   2               1705728300          9.485316e-02         9.622410e-02      0.014453
14       user  latency-90         2                   1               1705728060          6.565824e-03         6.632846e-03      0.010208
15      carts      socket         2                   1               1705729080          2.386000e+01         2.410000e+01      0.010059
16     orders      socket         2                   1               1705728840          1.994667e+01         2.014286e+01      0.009836
17      carts  latency-90         1                   2               1705728900          1.978317e+00         1.992000e+00      0.006917
18  catalogue    workload         2                   1               1705728060          4.244150e+00         4.270100e+00      0.006114
19       user         cpu         2                   1               1705728060          8.639787e-01         8.674499e-01      0.004018, 31, 50)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```